In [1]:
import warnings
warnings.filterwarnings('ignore')

# 데이터 처리 및 임베딩 기법

# 라이브러리 설치

## 설치하는 라이브러리의 역할  

데이터 추출 및 전처리(parsing)  
`pypdf`: PDF 파일에서 텍스트를 읽고 페이지를 분할하거나 합치는 등 PDF 데이터를 다루는 라이브러리  
`bs4(BeautifulSoup4)`: HTML이나 XML 파일(웹 페이지)에서 원하는 데이터를 쉽게 추출하는 웹 크롤링(스크레이핑) 라이브러리  
`jq`: 복잡한 JSON 형식의 데이터를 필터링하고 구조화하는 데 특화된 라이브러리  

자연어 처리 및 토큰화(NLP & Tokenization)  
`tiktoken`: OpenAI 모델이 텍스트를 처리하는 단위인 '토큰'으로 나누는 속도가 매우 빠른 토크나이저 라이브러리  
`transformers`: `Hugging Face`에서 제공하는 라이브러리로 BERT, GPT, Llama 등 많은 최신 AI 모델을 불러오는 라이브러리  
`langdetect`: 입력된 텍스트가 한국어인지, 영어인지 등 언어를 자동으로 감지해주는 라이브러리  

LangChain 생태계  
`langchain_experimental`: LangChain의 새로운 기능이나 실험적인 코드가 담겨있는 라이브러리  
`langchain_huggingface`: `Hugging Face`에 올라온 모델들을 LangChain 프레임워크 내에서 쉽게 쓸 수 있도록 연결하는 라이브러리  
`langchain_ollama`: 로컬 PC에서 AI돌리는 `Ollama`를 LangChain과 연동하여 오프라인 AI 서비스를 만들 때 사용하는 라이브러리  

In [2]:
# !pip install pypdf bs4 jq tiktoken transformers langdetect langchain_experimental langchain_huggingface langchain_ollama

# 환경 설정

## .env 환경 변수

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

## 기본 라이브러리

In [4]:
import os, json
from glob import glob
from pprint import pprint
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 다양한 형식의 문서 처리하기

## PDF 문서 가져오기

PDF 파일에서 텍스트를 추출(페이지별로 구분하여)을 읽어들이기(문서 객체로 변환) 위해서 PyPDFLoader를 import 한다.  
pypdf 라이브러리가 설치되어있어야 정상적으로 동작한다.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [6]:
# PyPDFLoader 클래스의 생성자로 읽어들일 PDF 파일의 경로와 이름을 넘겨서 PyPDFLoader 클래스 객체를 생성한다.
pdf_loader = PyPDFLoader(file_path='./data/transformer.pdf')

# PyPDFLoader 객체에서 load() 메소드를 실행해서 실제 PDF 파일에서 텍스트를 불러온다.
# 이때, PDF 파일의 각 페이지를 텍스트로 추출하여 Document 객체가 저장된 리스트 형태로 불러오고 각 Document 객체 내부에는 page_content(페이지 내부의 텍스트)와
# metadata(파일 이름, 페이지 번호 등) 정보가 저장되어 있다.
pdf_docs = pdf_loader.load()
print(len(pdf_docs))
print(type(pdf_docs))
print(pdf_docs)

15
<class 'list'>
[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/transformer.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kais

In [7]:
pdf_docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/transformer.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\n

In [8]:
pdf_docs[0].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': './data/transformer.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}

In [9]:
pdf_docs[0].page_content

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence

## 웹 문서 가져오기

사용자 에이전트(USER_AGENT) 식별 정보를 설정한다.  
USER_AGENT는 웹 브라우저나 프로그램을 통해 웹사이트에 접속할 때 '내가 어떤 프로그램으로 접속했는지' 웹 서버에 알려주는 식별 정보이다.  
최근 많은 웹사이트 API 서버는 자동화된 크롤러, 봇의 접근을 방지하거나 보안 정책에 의해 USER_AGENT 헤더가 비어있는 요청을 차단하기 때문에 'MyLLMApp/1.0'와 같이 임의의 이름, 버전 문자열을 지정함으로써 웹 서버에 요청시 차단되지 않고 정상적으로 데이터를 로드할 수 있도록 안전 장치를 마련한 것이다.

In [10]:
os.environ['USER_AGENT'] = 'MyLLMApp/1.0'

특정 웹 페이지의 내용을 읽어 텍스트를 추출해서 읽어들이기(문서 객체로 변환) 위해서 WebBaseLoader를 import 한다.  
bs4 라이브러리가 설치되어있어야 정상적으로 동작한다.

In [11]:
from langchain_community.document_loaders import WebBaseLoader

In [12]:
# WebBaseLoader 클래스의 생성자로 읽어들일 웹 문서의 주소를 넘겨서 WebBaseLoader 클래스 객체를 생성한다.
web_loader = WebBaseLoader(['https://python.langchain.com/', 'https://js.langchain.com/'])

# WebBaseLoader 객체에서 load() 메소드를 실행해서 실제 웹 문서의 텍스트 읽어온다.
# 이때, 웹 문서의 텍스트를 추출하여 Document 객체가 저장된 리스트 형태로 불러오고 각 Document 객체 내부에는 page_content(웹 문서의 텍스트)와 
# metadata(주소, 타이틀, 설명 등) 정보가 저장되어 있다.
web_docs = web_loader.load()
print(len(web_docs))
print(type(web_docs))
print(web_docs)

2
<class 'list'>
[Document(metadata={'source': 'https://python.langchain.com/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMess

In [13]:
web_docs[0]

Document(metadata={'source': 'https://python.langchain.com/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-ter

In [14]:
web_docs[0].metadata

{'source': 'https://python.langchain.com/',
 'title': 'LangChain overview - Docs by LangChain',
 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.',
 'language': 'en'}

In [15]:
web_docs[0].page_content

'LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringMCPHuman-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIProductionDeploymentObservabilityOn 

## JSON 문서 가져오기

JSON 또는 JSONL 파일의 내용을 읽어서 구조를 분석한 후 텍스트를 추출해서 읽어들이기(문서 객체로 변환) 위해서 JSONLoader를 import 한다.  
jq 라이브러리가 설치되어있어야 정상적으로 동작한다.

In [16]:
from langchain_community.document_loaders import JSONLoader

In [17]:
# JSONLoader 클래스의 생성자로 읽어들일 JSON 파일의 경로와 이름, JSON 파일의 데이터 구조를 넘겨서 JSONLoader 클래스 객체를 생성한다.
json_loader = JSONLoader(
    # 읽어들일 JSON 파일의 경로와 파일 이름을 지정한다.
    file_path='./data/kakao_chat.json',
    # jq 라이브러리를 사용해서 전체 데이터에서 어디를 읽을지 지정한다.
    # '전체 데이터의 messages라는 리스트([])' 안의 딕셔너리에서 'content'라는 key에 할당된 value만 가져오라는 의미이다. 대화 내용만 추출한다.
    jq_schema='.messages[].content',
    # text_content 속성은 추출한 데이터를 단순 문자열로 취급 여부를 설정한다.
    # 기본값(False)을 사용하면 jq_schema를 지정해서 추출한 데이터를 JSON 데이터로 취급한다.
    # True를 사용하면 jq_schema로 추출된 데이터가 '단순 문자열' 형태일 때 추가 가공 없이 원본 문자열 그대로 사용하게 한다. 단순 문자열로 취급한다.
    text_content=True,
)

# JSONLoader 객체에서 load() 메소드를 실행해서 JSON 문서의 텍스트 읽어온다.
# 이때, JSON 파일에서 jq_schema로 지정한 텍스트를 추출하여 Document 객체가 저장된 리스트 형태로 불러오고 각 Document 객체 내부에는 page_content(JSON 문서의 
# 텍스트)와 metadata(JSON 파일 경로, 일련 번호) 정보가 저장되어 있다.
json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 4}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 5}, page_content='네, 모두 준비했습니다. 회의 때 뵙겠습니다 :)')]


In [18]:
json_docs[0]

Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.')

In [19]:
json_docs[0].metadata

{'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json',
 'seq_num': 1}

In [20]:
json_docs[0].page_content

'안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'

이전 코드에서 jq_schema와 text_content를 수정했다.

In [21]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.json',
    # 이전 코드는 '.messages[].content'였지만 '.content'를 제거했다.
    # '.messages[].content'는 messages 리스트의 딕셔너리에서 'content'라는 key에 할당된 value를 가져오라는 의미였지만 '.content'를 제거하면 messages 리스트의
    # 모든 내용(sender, timestamp, content)을 한 덩어리로 가져온다.
    jq_schema='.messages[]',
    # 가져올 데이터가 단순 문자열이 아니다. JSON 형태의 데이터이다.
    text_content=False,
)

# 각 Document 객체의 'page_content'에는 단순 메시지만 들어가는 것이 아니라 {"sender": "홍길동", "timestamp": ..., "content": ...}와 같은 JSON 구조 전체가
# 텍스트로 들어간다.
json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='{"sender": "\\uae40\\ucca0\\uc218", "timestamp": "2023-09-15 09:30:22", "content": "\\uc548\\ub155\\ud558\\uc138\\uc694 \\uc5ec\\ub7ec\\ubd84, \\uc624\\ub298 \\ud68c\\uc758 \\uc2dc\\uac04 \\ud655\\uc778\\ucc28 \\uc5f0\\ub77d\\ub4dc\\ub9bd\\ub2c8\\ub2e4."}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2}, page_content='{"sender": "\\uc774\\uc601\\ud76c", "timestamp": "2023-09-15 09:31:05", "content": "\\ub124, \\uc548\\ub155\\ud558\\uc138\\uc694. \\uc624\\ud6c4 2\\uc2dc\\uc5d0 \\ud558\\uae30\\ub85c \\ud588\\uc5b4\\uc694."}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3}, page_content='{"sender": "\\ubc15\\ubbfc\\uc218", "timestamp": "2023-09-15 09:32:18", "content": "\\ud655\\uc778\\ud588\\uc2b5\\ub2c8\\ub2e4. 

text_content 속성의 속성값을 False로 지정해서 읽어들이면 한글이 깨져(유니코드 문자로) 보이는 현상이 발생된다.

In [22]:
# 유니코드로 인코딩된 데이터들을 다시 한글로 디코딩된 데이터로 저장할 리스트를 선언한다.
decoded_json_docs = []

for doc in json_docs:
    # print(type(doc))
    # page_content에 문자열 형태로 들어있는 JSON 데이터를 파이썬의 딕셔너리 형태로 변환한다.
    # json 라이브러리의 loads() 메소드로 파이썬의 딕셔너리 형태로 바꿔주면 유니코드 형태로 보이던 문자열이 실제 한글로 해석된다.
    decoded_data = json.loads(doc.page_content)
    # print(type(decoded_data))
    # print(decoded_data)
    
    decoded_json_docs.append({
        # 기존의 metadata는 그대로 유지한다.
        'metadata': doc.metadata,
        # 유니코드가 한글로 해석된 문자열을 추가한다.
        # 'page_content': decoded_data
        # 예전 버전으로 작성된 코드는 아래와 같은 코드를 사용하는 경우가 있다.
        # json 라이브러리의 dumps() 메소드의 ensure_ascii 속성의 속성값을 False로 지정해야 유니코드가 한글로 해석된 문자열이 추가된다.
        'page_content': json.dumps(decoded_data, ensure_ascii=False)
    })
    
print(len(decoded_json_docs))
print(type(decoded_json_docs))
print(decoded_json_docs)

5
<class 'list'>
[{'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, 'page_content': '{"sender": "김철수", "timestamp": "2023-09-15 09:30:22", "content": "안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다."}'}, {'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2}, 'page_content': '{"sender": "이영희", "timestamp": "2023-09-15 09:31:05", "content": "네, 안녕하세요. 오후 2시에 하기로 했어요."}'}, {'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3}, 'page_content': '{"sender": "박민수", "timestamp": "2023-09-15 09:32:18", "content": "확인했습니다. 회의실은 어디인가요?"}'}, {'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 4}, 'page_content': '{"sender": "이영희", "timestamp": "2023-09-15 09:33:40", "content": "3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!"}'}, {'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\

텍스트 내용(page_content)과 부가 정보(metadata)를 하나로 묶어주는 LangChain의 표준 데이터 규격인 Document를 사용하기 위해 import 한다.  
딕셔너리 형태로 만들어준 데이터를 LangChain의 다른 기능(텍스트 분할, 벡터 저장 등)과 호환되도록 LangChain의 표준 데이터 규격인 Document 객체로 변환한다.

In [23]:
from langchain_core.documents import Document

In [24]:
# 딕셔너리 형태의 데이터를 Document 클래스 객체로 변환한 데이터로 저장할 리스트를 선언한다.
decoded_json_docs = []

for doc in json_docs:
    decoded_data = json.loads(doc.page_content)
    # 유니코드를 한글로 디코딩한 새로눈 Document 클래스 객체를 생성한다.
    # 한글이 유니코드 형태로 깨지지 않도록 ensure_ascii=False 속성을 설정해서, 한글로 디코딩된 내용의 문자열을 저장한다.
    document_obj = Document(metadata=doc.metadata, page_content=json.dumps(decoded_data, ensure_ascii=False))
    decoded_json_docs.append(document_obj)
    
print(len(decoded_json_docs))
print(type(decoded_json_docs))
print(decoded_json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='{"sender": "김철수", "timestamp": "2023-09-15 09:30:22", "content": "안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다."}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2}, page_content='{"sender": "이영희", "timestamp": "2023-09-15 09:31:05", "content": "네, 안녕하세요. 오후 2시에 하기로 했어요."}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3}, page_content='{"sender": "박민수", "timestamp": "2023-09-15 09:32:18", "content": "확인했습니다. 회의실은 어디인가요?"}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 4}, page_content='{"sender": "이영희", "timestamp": "2023-09-15 09:33:40", "content": "3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!"}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\work

In [25]:
decoded_json_docs[0]

Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='{"sender": "김철수", "timestamp": "2023-09-15 09:30:22", "content": "안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다."}')

In [26]:
decoded_json_docs[0].metadata

{'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json',
 'seq_num': 1}

In [27]:
decoded_json_docs[0].page_content

'{"sender": "김철수", "timestamp": "2023-09-15 09:30:22", "content": "안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다."}'

JSON 파일 안의 특정 필드(`content`)를 본문(`page_content`)으로 지정하고 다른 필드(`sender`, `timestamp`)들을 추출해서 메타데이터(`metadata`)로 합친다.

각 JSON 레코드를 읽을 때 메타데이터를 어떻게 구성할지 정의하는 함수

In [28]:
# JSONLoader 객체의 metadata_func 속성에서 실행할 함수
def metadata_func(record, metadata):
    # print(record, metadata)
    # 현재 처리하는 JSON 파일에서 읽어들인 데이터 1건(딕셔녀리)에서 'sender'라는 key에 할당된 value를 얻어와서 메타데이터에 추가한다.
    metadata['sender'] = record['sender']
    # 현재 처리하는 JSON 파일에서 읽어들인 데이터 1건(딕셔녀리)에서 'timestamp'라는 key에 할당된 value를 얻어와서 메타데이터에 추가한다.
    metadata['timestamp'] = record['timestamp']
    return metadata

In [29]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.json',
    # 전체 JSON 데이터 중에서 'messages' 리스트를 구성하는 딕셔너리에서 'content'라는 key에 할당된 value를 얻어온다.
    # jq_schema='.messages[].content',
    # 전체 JSON 데이터 중에서 'messages' 리스트를 얻어온다.
    jq_schema='.messages[]',
    # jq_schema를 이용해서 얻어온 리스트를 구성하는 딕셔너리에서 'content'라는 key에 할당된 value를 얻어온다. 본문(page_content)으로 사용한다.
    content_key='content',
    # JSON 레코드를 읽을 때 메타데이터를 어떻게 구성할지 정의하는 함수를 실행한다. 메타 데이터(metadata)를 만든다.
    # metadata_func 속성으로 실행할 함수를 지정하면 함수의 첫 번째 인수로 JSON 파일에서 읽어들인 page_content 전체(데이터 1건, 레코드)가 넘어가고 두 번째
    # 인수로 두 번째 인수로 metadata가 자동으로 넘어간다.
    metadata_func=metadata_func,
)

json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1, 'sender': '김철수', 'timestamp': '2023-09-15 09:30:22'}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2, 'sender': '이영희', 'timestamp': '2023-09-15 09:31:05'}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3, 'sender': '박민수', 'timestamp': '2023-09-15 09:32:18'}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 4, 'sender': '이영희', 'timestamp': '2023-09-15 09:33:40'}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 5, 'sender': '김철수'

In [30]:
json_docs[0]

Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1, 'sender': '김철수', 'timestamp': '2023-09-15 09:30:22'}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.')

In [31]:
json_docs[0].metadata

{'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json',
 'seq_num': 1,
 'sender': '김철수',
 'timestamp': '2023-09-15 09:30:22'}

In [32]:
json_docs[0].page_content

'안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'

일반적인 JSON 파일이 아니라, 한 줄에 하나의 JSON 객체가 저장된 JSONL(JSON Lines) 파일을 처리한다. 대용량 로그 데이터나 센서 데이터 또는 채팅 기록을 다룰 때 효율적인 방식이다.

In [33]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.jsonl',
    # 이전의 '.messages[]'와 달리 JSONL 파일은 한 줄이 하나의 레코드이므로 리스트를 순회할 필요 없이 바로 key 이름을 지정한다.
    jq_schema='.content',
    # json_lines 속성의 속성값을 True로 지정해서, JSONLoader에게 '한 줄 한 줄이 개별 JSON 데이터'라고 알려준다.
    # 이 속성이 있어야 JSONL 파일을 정상적으로 읽을 수 있다.
    json_lines=True,
)

json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 1}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 2}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 3}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 4}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 5}, page_content='네, 모두 준비했습니다. 회의 때 뵙겠습니다 :)')]


개별 줄 전체(`.`)를 대상으로 하되 그중 특정 키(`content`)만 본문(`page_content`)으로 불러오는 방식이다.

In [34]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.jsonl',
    # '.'은 현재 줄의 전체를 대상으로 하라는 의미이다.
    jq_schema='.',
    content_key='content',
    json_lines=True,
)

json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 1}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 2}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 3}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 4}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 5}, page_content='네, 모두 준비했습니다. 회의 때 뵙겠습니다 :)')]


이전의 JSON 파일을 읽어서 메타데이터를 추가했던 것 처럼 JSONL 파일을 읽어서 메타데이터를 추가한다.

In [35]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.jsonl',
    jq_schema='.',
    content_key='content',
    metadata_func=metadata_func,
    json_lines=True,
)

json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 1, 'sender': '김철수', 'timestamp': '2023-09-15 09:30:22'}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 2, 'sender': '이영희', 'timestamp': '2023-09-15 09:31:05'}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 3, 'sender': '박민수', 'timestamp': '2023-09-15 09:32:18'}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 4, 'sender': '이영희', 'timestamp': '2023-09-15 09:33:40'}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 5, 'sender': 

## CSV 문서 가져오기

CSV 파일의 내용을 읽어들이기(문서 객체로 변환) 위해서 CSVLoader를 import 한다.  

from langchain_community.document_loaders.csv_loader import CSVLoader  
예전에는 langchain_community.document_loaders`.csv_loader` 모듈을 사용했지만 버전이 올라가면서 모든 로더는 langchain_community.document_loaders 모듈에서 일괄 관리되면서 아래와 같이 사용한다.

In [36]:
from langchain_community.document_loaders import CSVLoader

In [37]:
# CSVLoader 클래스의 생성자로 읽어들일 CSV 파일의 경로와 이름을 필요에 따라서 인코딩 방식을 넘겨서 CSVLoader 클래스 객체를 생성한다.
# UnicodeDecodeError: 'cp949' codec can't decode byte 0xed in position와 같은 에러가 발생되면 encoding='utf-8' 속성을 지정한다.
csv_loader = CSVLoader(file_path='./data/kbo_teams_2023.csv', encoding='utf-8')

# CSVLoader 객체에서 load() 메소드를 실행해서 CSV 문서의 텍스트 읽어온다.
# Document 객체가 저장된 리스트 형태로 불러오고 각 Document 객체 내부에는 page_content(CSV 파일의 텍스트)와 metadata(CSV 파일 경로, 줄 번호) 정보가 저장되어 
# 있다.
csv_docs = csv_loader.load()
print(len(csv_docs))
print(type(csv_docs))
print(csv_docs)

10
<class 'list'>
[Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 0}, page_content="Team: KIA 타이거즈\nCity: 광주\nFounded: 1982\nHome Stadium: 광주-기아 챔피언스 필드\nChampionships: 11\nIntroduction: KBO 리그의 전통 강호로, 역대 최다 우승 기록을 보유하고 있다. '타이거즈 스피릿'으로 유명하며, 양현종, 안치홍 등 스타 선수들을 배출했다. 광주를 연고로 하는 유일한 프로야구팀으로 지역 사랑이 강하다."), Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 1}, page_content='Team: 두산 베어스\nCity: 서울\nFounded: 1982\nHome Stadium: 잠실야구장\nChampionships: 6\nIntroduction: 2015년부터 2019년까지 5년 연속 한국시리즈에 진출한 강팀이다. 김태형 감독 체제에서 체계적인 선수 육성으로 주목받았으며, 정수빈, 김재환 등 핵심 선수들의 활약이 돋보인다.'), Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 2}, page_content="Team: SSG 랜더스\nCity: 인천\nFounded: 2000\nHome Stadium: 인천SSG랜더스필드\nChampionships: 5\nIntroduction: SK 와이번스에서 SSG 랜더스로 구단명을 변경했다. 2022년 와일드카드 결정전부터 한국시리즈까지 전승으로 우승을 차지하며 '노히트 행진'을 달성했다. 추신수, 김광현 등 스타 선수들이 포진해 있다."), Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 3}, page_content='Team: 삼성 라이온즈\nC

In [38]:
csv_docs[4]

Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 4}, page_content='Team: LG 트윈스\nCity: 서울\nFounded: 1982\nHome Stadium: 잠실야구장\nChampionships: 2\nIntroduction: 서울을 연고로 하는 인기 구단으로, 오랜 기간 우승이 없었으나 2023년 29년 만에 한국시리즈 우승을 차지했다. 김현수, 고우석 등 국가대표급 선수들이 포진해 있으며, 탄탄한 마운드가 강점이다.')

In [39]:
csv_docs[4].metadata

{'source': './data/kbo_teams_2023.csv', 'row': 4}

In [40]:
csv_docs[4].page_content

'Team: LG 트윈스\nCity: 서울\nFounded: 1982\nHome Stadium: 잠실야구장\nChampionships: 2\nIntroduction: 서울을 연고로 하는 인기 구단으로, 오랜 기간 우승이 없었으나 2023년 29년 만에 한국시리즈 우승을 차지했다. 김현수, 고우석 등 국가대표급 선수들이 포진해 있으며, 탄탄한 마운드가 강점이다.'

# 텍스트 분할 전략

## RecursiveCharacterTextSplitter를 사용해서 문서 조각(chunk)으로 분할하기

LangChain에서 제공하는 고급 텍스트 분할 도구로 텍스트를 재귀적으로 분할하여 더 자연스러운 문서 조각(청크)를 생성한다.  
순차적 구분자 적용 순서: `\n\n`(문단, 단락 단위- 최우선) => `\n`(줄 단위) => `.`(문장 단위) => `' '`(공백, 단어 단위) => `''`(글자 단위 - 최후의 수단)  
CharacterTextSplitter보다 더 엄격하게 크기를 준수하려는 경향이 있다. => 01_LangChain의_주요_RAG_컴포넌트 예제에서 설명했다.

구분자를 지정해서(여러 개) 텍스트를 재귀적으로 분할하여 문서 조각(청크)을 생성하기 위해서 RecursiveCharacterTextSplitter를 import 한다.

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [42]:
# 문맥을 최대한 보존하면서 문서를 자르는 도구로 단순히 글자 수로만 자르는 것이 아니라 문단이나 줄바꿈 같은 구분자를 기준으로 자연스럽게 잘라준다.
# RecursiveCharacterTextSplitter 클래스의 생성자로 청크의 크기, 문맥을 유지하기 위해 겹치는 크기, 구분자, 길이를 재는 기준을 넘겨서 객체를 만든다.
text_splitter = RecursiveCharacterTextSplitter(
    # 텍스트를 재귀적으로 분할할 문서 조각(청크)의 크기를 1,000으로 지정한다.
    chunk_size=1000,
    # 텍스트를 문서 조각으로 자를때 문맥이 끊기는 것을 방지하기 위해서 앞의 문서 조각의 끝부분 내용을 뒤의 문서 조각이 포함해서 겹치는 크기를 200으로 지정한다.
    chunk_overlap=200,
    # \n\n => \n => '.' => ' ' => '' 순서로 텍스트를 문서 조각으로 자르는 단위를 지정한다. 여러개의 구분자를 지정할 수 있다.
    # 먼저 문단(\n\n) 단위로 잘라보고, 그래도 1,000자가 넘으면 줄바꿈(\n) 단위로 자른다. 이렇게 하면 문서 중간이 잘리는 현상을 최소화할 수 있다.
    separators=['\n\n', '\n'],
    # 문서 조각의 길이를 재는 함수는 지정한다. 필요에 따라 tiktoken을 사용해 토큰 수를 기준으로 잴 수도 있다.
    length_function=len,
)

# RecursiveCharacterTextSplitter 객체의 split_documents() 메소드를 이용해서 텍스트를 문서 조각으로 자른다.
texts = text_splitter.split_documents(pdf_docs)
print(len(texts))
print([len(text.page_content) for text in texts])

52
[984, 910, 975, 452, 930, 995, 902, 907, 994, 382, 923, 951, 216, 917, 996, 841, 988, 913, 905, 868, 928, 965, 943, 997, 196, 974, 971, 946, 930, 986, 943, 918, 734, 958, 946, 945, 617, 982, 988, 994, 624, 944, 909, 941, 914, 986, 925, 927, 847, 812, 815, 818]


In [43]:
# 각 문서 조각의 겹치는 부분을 확인한다.
for i in range(len(texts[:3])):
    print(texts[i].page_content[-200:])
    print('*' * 100)
    print(texts[i + 1].page_content[:200])
    print('=' * 100)
    print()

the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
****************************************************************************************************
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine transla

the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.
****************************************************************************************************
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training dat

## CharacterTextSplitter와 정규 표현식을 사용해서 문서 조각(chunk)으로 분할하기

정규 표현식을 사용하면 특정 패턴을 기반으로 텍스트를 더 정확하게 분할할 수 있어서 구조화된 텍스트나 특정 형식의 문서에 유용하다.  
제1조, 제1장, 문장 단위(마침표, 느낌표, 물음표)로 끝나는 문장에 활용한다.

기본적인 텍스트 분할기로 구분자를 지정해서(1개) 텍스트를 분할하여 문서 조각(청크)을 생성하기 위해서 CharacterTextSplitter를 import 한다.

In [44]:
from langchain_text_splitters import CharacterTextSplitter

텍스트를 단순히 글자 수로 자르는 것이 아니고 문장 부호(`.`, `!`, `?`)를 기준으로 의미있는 문장 단위로 정규 표현식을 사용해서 나눈다.

정규 표현식 `r'(?<=[.!?])\s+'`는 마침표(.), 물음표(?), 느낌표(!) 바로 뒤에 나오는 하나 이상의 공백 문자를 찾는 패턴이다.  
`(?<=...)`: 후방 탐색을 의미하고 조건(`[.!?]`)이 일치하는 위치를 찾되, 해당 문자를 매칭 결과에 포함시키지 않는 역할을 합니다.  
`[.!?]`: 마침표(.), 물음표(?), 느낌표(!) 중 하나를 의미한다.  
`\s+`: 공백 문자들, `\s`는 스페이스(공백), 탭(\t), 줄바꿈(\n) 등 모든 공백 문자를 의미하고 `+`는 직전의 문자가 1개 이상 연속하여 존재하는 경우를 뜻한다.

검색 조건으로만 사용되고 실제 치환이나 분할 시 잘려나가지 않습니다.

In [45]:
# CharacterTextSplitter 클래스의 생성자로 청크의 크기, 문맥을 유지하기 위해 겹치는 크기, 구분자를 넘겨서 객체를 만든다.
text_splitter = CharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    # 마침표, 느낌표, 물음표 중 하나가 나오고, 그 뒤에 공백이 이어지는 위치를 찾아서 나누는 구분자를 지정한다.
    separator=r'(?<=[.!?])\s+',
    # separator로 정규 표현식을 사용하려면 is_separator_regex 속성의 속성값을 True로 저정해야 한다.
    is_separator_regex=True,
    # 분할 기준이 된 문장 부호를 버리지 않고 문장 끝에 그대로 붙여둔다.
    keep_separator=True
)

texts = text_splitter.split_documents(json_docs)
print(len(texts))
print([len(text.page_content) for text in texts])

Created a chunk of size 11, which is longer than the specified 10
Created a chunk of size 13, which is longer than the specified 10


9
[31, 9, 15, 7, 11, 11, 27, 13, 13]


In [46]:
for i in range(len(texts[:3])):
    print(texts[i].page_content[-200:])
    print('*' * 100)
    print(texts[i + 1].page_content[:200])
    print('=' * 100)
    print()

안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.
****************************************************************************************************
네, 안녕하세요.

네, 안녕하세요.
****************************************************************************************************
오후 2시에 하기로 했어요.

오후 2시에 하기로 했어요.
****************************************************************************************************
확인했습니다.



## 토큰 수를 기반으로 문서 조각(chunk)으로 분할하기

토큰 기반 분할은 LLM의 토큰 제한을 고려할 때 유용하며 각 문서 조각(청크)이 특정 토큰 수를 초과하지 않도록 조절 가능하다.  
tiktoken, transformers 라이브러리가 설치되어있어야 정상적으로 동작한다.

In [47]:
print(len(pdf_docs[0].page_content))
print(pdf_docs[0].page_content)

2857
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. 

tiktoken 라이브러리(OpenAI의 토큰 계산기)를 사용해 텍스트를 나누는 객체를 선언해서 일반적인 글자 수가 아니라, 실제 LLM이 받아들이는 토큰 수를 기준으로 자른다.

주요 인코더 비교  
`특징                 cl100k_base                           o200k_base                              `  
`적용 모델            GPT-3.5-Turbo, GPT-4, GPT-4-Turbo     GPT-4o, GPT-4o-mini                     `  
`어휘집 크기          약 100,000(100k)                      약 200,000(200k)                        `  
`한국어 토큰 효율     기준점                                약 1.5 ~ 2배 이상 감소(동일 텍스트 대비)`  
`BPE 알고리즘         Byte Pair Encoding                    Byte Pair Encoding(다국어 최적화)       `

RecursiveCharacterTextSplitter 라이브러리의 from_tiktoken_encoder() 메소드를 사용해서 토큰 수를 기반으로 분할한다.

In [48]:
# from_tiktoken_encoder() 메소드는 LLM의 토큰 수를 기준으로 텍스트를 측정 및 분할한다.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    # 토큰을 계산할 인코딩 방식(알고리즘)을 지정한다.
    encoding_name='cl100k_base',
    # 특정 모델 이름을 직접 지정할 수도 있다.
    # model_name='gpt-4o-mini'를 사용하면 'encoding_name' 속성 설정을 생략해도 해당 모델에 맞게 자동으로 설정된다.
    # model_name='gpt-4o-mini',
    # 하나의 문서 조각(청크)에 들어갈 최대 문자수를 의미하는 것이 아니고 최대 토큰 수를 300개로 설정한다.
    # LLM은 한 번에 읽을 수 있는 토큰 양의 한계가 있으므로, 효율적인 정보 검색을 위해 적절한 크기(보통 300 ~ 1000)로 잘라준다.
    chunk_size=300,
    # 나눠진 청크들 사이에 중복 내용을 얼마나 둘지 결정한다. 보통 문맥 연결을 위해 20 ~ 50 정도를 주기도 한다.
    chunk_overlap=0,
)

chunks = text_splitter.split_documents(pdf_docs[:1])
print(len(chunks))
print([len(chunk.page_content) for chunk in chunks])

3
[1143, 1374, 338]


In [49]:
for i in range(len(chunks) - 1):
    print(chunks[i].page_content[-50:])
    print('*' * 100)
    print(chunks[i + 1].page_content[:50])
    print('=' * 100)
    print()

ng more parallelizable and requiring significantly
****************************************************************************************************
less time to train. Our model achieves 28.4 BLEU o

countless long days designing various parts of and
****************************************************************************************************
implementing tensor2tensor, replacing our earlier 



## tiktoken 토크나이저로 임베딩 하기

OpenAI에서 개발한 고속 토큰화 라이브러리인 tiktoken을 사용하기 위해 import 한다.

In [50]:
import tiktoken

tiktoken 라이브러리의 get_encoding() 메소드를 사용해서 토큰화 한다.

In [51]:
# get_encoding() 메소드의 인수로 토큰을 계산할 인코딩 방식을 넘겨서 토크나이저 객체를 생성한다.
tokenizer = tiktoken.get_encoding('cl100k_base')
# encoding_for_model() 메소드의 인수로 특정 모델을 넘겨서 토크나이저 객체를 생성한다. 해당 모델에 맞게 자동으로 설정된다.
# tokenizer = tiktoken.encoding_for_model('gpt-4o-mini')

In [52]:
for chunk in chunks:
    # 인코딩(텍스트 => 숫자)
    # 문서 조각(청크, chunk.page_content)를 tiktoken 라이브러리를 이용한 토크나이저로 정수 리스트로 변환한다.
    # AI 모델은 텍스트를 직접 읽지 못하고, 이 과정을 통해 변환된 숫자를 입력받는다.
    # encode() 메소드의 인수로 문서 조각을 넘겨서 토큰화 한다.
    tokens = tokenizer.encode(chunk.page_content)
    # 앞서 설정한 chunk_size=300에 맞게 토큰으로 잘 나눠졌는지 변환된 토큰의 개수를 출력한다.
    print(len(tokens))
    # 변환된 토큰 중 앞의 10개만 출력해본다.
    print(tokens[:10])
    
    # 디코딩(숫자 => 텍스트)
    # 인코딩된 숫자가 어떤 단어나 문자에 대응하는지 확인해본다.
    # decode() 메소드의 인수로 토큰을 리스트 형태로 넘겨서 원래 텍스트로 복원한다.
    token_strings = [tokenizer.decode([token]) for token in tokens[:10]]
    # 복원된 10개의 토큰에 해당되는 문자열을 출력해본다.
    print(token_strings)
    print('-' * 100)

287
[36919, 291, 6300, 63124, 374, 3984, 11, 5195, 22552, 25076]
['Provid', 'ed', ' proper', ' attribution', ' is', ' provided', ',', ' Google', ' hereby', ' grants']
----------------------------------------------------------------------------------------------------
291
[1752, 892, 311, 5542, 13, 5751, 1646, 83691, 220, 1591]
['less', ' time', ' to', ' train', '.', ' Our', ' model', ' achieves', ' ', '28']
----------------------------------------------------------------------------------------------------
84
[95574, 287, 16000, 17, 47211, 11, 25935, 1057, 6931, 2082]
['implement', 'ing', ' tensor', '2', 'tensor', ',', ' replacing', ' our', ' earlier', ' code']
----------------------------------------------------------------------------------------------------


## Hugging Face 토크나이저로 임베딩 하기

Hugging Face의 transformers 라이브러리의 모델의 이름만 입력하면, 그 모델이 만들어질 때 사용되었던 도구(BERT, GPT, Llama 등)가 무었인지 자동으로 판단하여 적절한 토크나이저를 불러오는 AutoTokenizer를 사용하기 위해 import 한다.

In [53]:
from transformers import AutoTokenizer

from_pretrained() 메소드의 인수로 지정할 수 있는 모델의 종류

한국어 텍스트 처리 및 분류: `klue/bert-base`, monologg/kobert, skt/kobert-base-v1, kykim/bert-kor-base  
생성형 LLM: Qwen/Qwen2.5-7B-Instruct, meta-llama/Meta-Llama-3-8B-Instruct, google/gemma-2-2b-it  
문장 임베딩 및 검색: `BAAI/bge-m3`, jhgan/ko-sroberta-multitask, intfloat/multilingual-e5-large

`BAAI/bge-m3` 모델은 베이징 인공지능 연구원(Beijing Academy of Artificial Intelligence, BAAI)에서 공개한 임베딩 모델로, 현재 RAG 스택 및 정보 검색 영역에서 표준처럼 사용되는 대표적인 오픈소스 다국어 임베딩 모델이다.  
M3는 Multi-Linguality(우수한 다국어 지원), Multi-Functionality(하이브리드 검색 통합), Multi-Granularity(긴 문맥 지원)를 의미한다.

In [54]:
# from_pretrained() 메소드의 인수로 모델을 넘겨서 그 모델이 만들어질 때 사용된 토크나이저를 불러온다.
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-m3')
tokenizer

XLMRobertaTokenizer(name_or_path='BAAI/bge-m3', vocab_size=250002, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	250001: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=False, special=True),
})

'안녕하세요. 반갑습니다.'라는 문자열을 'BAAI/bge-m3' 모델에 사용한 토크나이저에 입력해서 숫자 리스트로 변환한다.  

작동 순서  
① 텍스트 정규화: 띄어쓰기나 특수문자를 모델이 약속한 규칙대로 정리한다.  
② 토큰화: 문장을 의미있는 최소 단위로 나눈다.  
③ ID 매핑(숫자화): 나눠진 각 토큰을 미리 만들어진(모델이 제공하는) 단어 사전에서 찾아 해당되는 고유한 숫자로 바꾼다.  
④ 특수 토큰 추가: 문장의 시작과 끝을 알리는 특수 토큰(`<s>`, `</s>`)을 앞뒤에 붙인다.

In [55]:
tokens = tokenizer.encode('안녕하세요. 반갑습니다.')
tokens

[0, 107687, 5, 20451, 54272, 16367, 5, 2]

In [56]:
# convert_ids_to_tokens() 메소드의 인수로 토큰을 넘겨서 토크나이저가 내부적으로 가지고 있는 단어 사전을 참조하여 각 숫자에 대응하는 글자를 찾아준다.
print(tokenizer.convert_ids_to_tokens(tokens))

['<s>', '▁안녕하세요', '.', '▁반', '갑', '습니다', '.', '</s>']


In [57]:
# decode() 메소드의 인수로 토큰을 넘기면 토크나이저의 단어 사전을 참조하여 다시 글자로 바꾸고 하나로 합쳐준다.
# 단순히 숫자만 글자로 바꾸는 게 아니라, 특수 기호나 띄어쓰기 등의 복원 규칙을 적용하여 처음에 입력했던 형태와 거의 흡사하게 문장을 재구성한다.
# decode() 메소드를 사용하면 문장 앞뒤에 붙은 <s>(시작), </s>(끝) 같은 특수 토큰까지 모두 보여준다.
print(tokenizer.decode(tokens))
# 특수 토큰을 보기 싫으면 skip_special_tokens 속성의 속성값을 True로 지정하면 순수한 텍스트만 추출할 수 있다.
print(tokenizer.decode(tokens, skip_special_tokens=True))

<s> 안녕하세요. 반갑습니다.</s>
안녕하세요. 반갑습니다.


from_tiktoken_encoder() 메소드는 OpenAI 방식으로 임베딩을 실행하고 from_huggingface_tokenizer() 메소드는 Hugging Face 방식으로 임베딩을 실행한다.

In [58]:
# from_huggingface_tokenizer() 메소드는 Hugging Face의 AutoTokenizer 클래스의 from_pretrained() 메소드로 가져온 임베딩 모델을 사용한다.
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer, # BAAI/bge-m3 모델에 사용한 토크나이저로 토큰화 한다.
    chunk_size=300,
    chunk_overlap=0,
)

chunks = text_splitter.split_documents(pdf_docs[:1])
print(len(chunks))
print([len(chunk.page_content) for chunk in chunks])

3
[1217, 1300, 338]


In [59]:
for chunk in chunks:
    tokens = tokenizer.encode(chunk.page_content)
    print(len(tokens))
    print(tokens[:10])
    token_strings = tokenizer.convert_ids_to_tokens(tokens[:10])
    print(token_strings)
    print(tokenizer.decode(tokens, skip_special_tokens=True))
    print('-' * 100)

298
[0, 123089, 71, 27798, 99, 179236, 83, 62952, 4, 1815]
['<s>', '▁Provide', 'd', '▁proper', '▁at', 'tribution', '▁is', '▁provided', ',', '▁Google']
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propos

## 맥락을 기반으로 분할하기

의미적 유사성에 기반하여 분할하며 문장들 사이의 임베딩 차이를 기반으로 작동한다.

LangChain의 실험적 기능(experimental) 패키지에서 문장 간의 의미적 유사도를 계산하여, 내용이 달라지는 지점에서 문서 조각으로 나누는 SemanticChunker를 사용하기 위해 import 한다.

In [60]:
from langchain_experimental.text_splitter import SemanticChunker

기존에 사용하던 `글자 수`나 `토큰 수`를 기준으로 하는 분할 방식에서 벗어나. `문장의 의미(semantic)`가 변하는 지점을 AI가 스스로 판단하여 문서를 나눈다.

In [61]:
# 의미 기반 텍스트 분할기 객체를 생성한다.
text_splitter = SemanticChunker(
    # 의미를 비교할 때 사용할 임베딩 모델을 지정한다.
    # 모든 문장을 임베딩(숫자화)한다. => 앞 문장과 뒷 문장의 유사도를 계산한다. => 유사도가 멀어지면 '여기부터는 다른 내용이구나'라고 판단하여 자른다.
    embeddings=OpenAIEmbeddings(model='text-embedding-3-small'),
    # 문장을 자를 기준점(breakpoint)을 결정하는 알고리즘을 지정한다.
    # 'gradient'는 문장들 사이의 유사도 변화율(기울기)을 분석한다. 단순히 유사도가 낮은 곳을 찾는 게 아니라, 의미가 갑자기 크게 변하는 지점(급경사)을 포착하여
    # 자르기 때문에 훨씬 정교한 분할이 가능하다.
    # 'gradient' 외에도 percentile(백분위) 방식과 standard_deviation(표준편차) 등의 옵션이 있다.
    breakpoint_threshold_type='gradient'
)

chunks = text_splitter.split_documents(pdf_docs[:1])
print(len(chunks))
print([len(chunk.page_content) for chunk in chunks])

2
[1739, 1117]


In [62]:
for chunk in chunks:
    tokens = tokenizer.encode(chunk.page_content)
    print(len(tokens))
    print(tokens[:10])
    token_strings = tokenizer.convert_ids_to_tokens(tokens[:10])
    print(token_strings)
    print(tokenizer.decode(tokens, skip_special_tokens=True))
    print('-' * 100)

419
[0, 123089, 71, 27798, 99, 179236, 83, 62952, 4, 1815]
['<s>', '▁Provide', 'd', '▁proper', '▁at', 'tribution', '▁is', '▁provided', ',', '▁Google']
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propos

# 문서 임베딩 모델

임베딩 모델이란?

텍스트를 벡터(텍스트의 의미를 숫자 배열로 나타낸 것) 표현으로 변환하는 모델로 텍스트를 벡터로 표현함으로써 의미적으로 가장 유사한 다른 텍스트를 찾는 수학적 연산을 수행한다.  
의미 기반 검색, 문서 분류, 텍스트 유사도 분석 등 다양한 자연어 처리 작업을 수행한다.  
OpenAI, Hugging Face, Ollama 등 다양한 임베딩 모델을 제공한다.

# OpenAI 임베딩 모델

OpenAI 임베딩 모델을 사용하려면 OpenAIEmbeddings 클래스를 사용한다.

장점  
&nbsp;&nbsp;&nbsp;▶ 높은 품질의 임베딩을 제공한다.  
&nbsp;&nbsp;&nbsp;▶ 다양한 언어와 도메인에 대해 잘 작동한다.  
&nbsp;&nbsp;&nbsp;▶ 지속적으로 업데이트되어 최신 기술을 반영한다.  
단점  
&nbsp;&nbsp;&nbsp;▶ API 사용에 비용이 발생된다.  
&nbsp;&nbsp;&nbsp;▶ 인터넷 연결이 필요하다.  
&nbsp;&nbsp;&nbsp;▶ 데이터가 외부 서버로 전송되므로 개인정보 보호 문제가 있을 수 있다.  

In [63]:
# 텍스트를 숫자로 변환(벡터화)하기 위해서 OpenAIEmbeddings 모델을 만든다.
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')
embeddings_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000286AE49DB10>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x00000286AE4BE650>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [64]:
# embedding_ctx_length 속성은 모델이 문맥을 파학하고 임베딩할 수 있는 최대 토큰의 개수를 얻어온다.
# 만약 문서가 이 길이보다 길다면 모델이 한 번에 처리하지 못해서 에러가 나거나 뒷 부분이 잘릴 수 있다. 이 길에 맞춰서 문서를 적절히 나눠야 한다.
embeddings_model.embedding_ctx_length

8191

embed_query() 메소드는 단일 문장을 임베딩 모델을 통해 숫자로 변환한다. 

In [65]:
embedded_query = embeddings_model.embed_query('인공지능이란 무엇인가요?')

print(len(embedded_query))
print(embedded_query)

1536
[-0.0224456787109375, 0.0222015380859375, 0.0003859996795654297, 0.00574493408203125, 0.0122528076171875, -0.044769287109375, -0.026275634765625, 0.035797119140625, -0.0025959014892578125, 0.01476287841796875, -0.0019893646240234375, 0.0009360313415527344, -0.004917144775390625, -0.0736083984375, 0.00899505615234375, -0.0154266357421875, -0.05987548828125, -0.02239990234375, 0.022491455078125, -0.06915283203125, -0.02923583984375, 0.023223876953125, -0.036865234375, 0.0041046142578125, 0.011077880859375, -0.052734375, 0.01071929931640625, 9.28044319152832e-05, -0.0108489990234375, -0.03424072265625, 0.0253143310546875, -0.018829345703125, -0.0015420913696289062, -0.058807373046875, 0.0498046875, -0.0034999847412109375, -0.0028629302978515625, -0.00855255126953125, -0.00460052490234375, 0.0228729248046875, -0.01197052001953125, 0.0379638671875, 0.0033168792724609375, 0.035400390625, -0.051361083984375, 0.034393310546875, -0.0285491943359375, 0.004730224609375, -0.01360321044921875,

embed_documents() 메소드는 여러 개의 문장을 임베딩 모델을 통해 숫자로 변환한다.

In [66]:
documents = [
    '인공지능은 컴퓨터 과학의 한 분야입니다.',
    '머신러닝은 인공지능의 하위 분야입니다.',
    '딥러닝은 머신러닝의 한 종류입니다.',
    '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
    '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.',
]
embedded_documents = embeddings_model.embed_documents(documents)

print(len(embedded_documents))
print(len(embedded_documents[0]))
print(embedded_documents)

5
1536
[[-0.0022869110107421875, 0.01218414306640625, -0.0024089813232421875, 0.0150146484375, 0.018402099609375, -0.045654296875, -0.003086090087890625, 0.046966552734375, -0.01824951171875, -0.031829833984375, 0.006900787353515625, 0.006969451904296875, -0.0185699462890625, -0.027099609375, 0.004978179931640625, -0.015533447265625, -0.0416259765625, 0.00998687744140625, 0.051544189453125, -0.0458984375, -0.01459503173828125, -0.0277862548828125, -0.0187225341796875, -0.0227203369140625, 0.00441741943359375, -0.03973388671875, 0.05029296875, 0.017730712890625, 0.0003154277801513672, -0.0233001708984375, 0.048919677734375, -0.0152587890625, -0.029937744140625, -0.06158447265625, 0.0205230712890625, 0.03546142578125, 0.002574920654296875, -0.00681304931640625, -0.005096435546875, 0.0247650146484375, 0.007568359375, 0.0274658203125, -0.0025768280029296875, 0.02606201171875, -0.01800537109375, 0.0054779052734375, -0.0258636474609375, 0.0036678314208984375, 0.0075225830078125, 0.0567321777

## 유사도 검사

질문의 의미를 파악하여 가장 연관성 높은 문장을 찾아내는 의미 검색(semantic search)을 실행한다.

코사인 유사도를 계산하기 위해서 cosine_similarity 라이브러리를 import 한다.  
코사인 유사도는 두 벡터 사이의 각도를 측정하여 방향이 얼마나 일치하는지(의미가 얼마나 비슷한지)를 0에서 1사이의 값을로 계산하고 1에 가까울수록 비슷한다.

In [67]:
from langchain_community.utils.math import cosine_similarity

In [68]:
import numpy as np

코사인 유사도를 계산하는 함수를 정의한다. 이 함수는 질문을 넣으면 질문과 가장 비슷한 문장을 찾아준다.

In [69]:
# query는 사용자 질문이고 embedded_documents는 query와 코사인 유사도를 검사할 2차원 벡터 집합이다.
def find_similar(query, embedded_documents):
    # 입력받은 질문(query)을 임베딩 벡터로 변환한다.
    query_embedding = embeddings_model.embed_query(query)
    # 질문 임베딩 벡터(query_embedding)는 1차원이고 embedded_documents는 2차원이므로 질문 임베딩 벡터를 2차원으로 만들어 코사인 유사도 검사를 실행한다.
    similarities = cosine_similarity([query_embedding], embedded_documents)[0]
    # print(similarities)
    # print(np.argmax(similarities))
    # 넘파이의 argmax() 메소드는 인수로 지정된 배열에서 최대값의 인덱스를 리턴한다.
    index = np.argmax(similarities)
    return documents[index], similarities[index]

In [70]:
queries = [
    '인공지능이란 무엇인가요?',
    '딥러닝과 머신러닝의 관계는 어떻게 되나요?',
    '컴퓨터가 이미지를 이해하는 방법은 무엇인가요?',
]

for query in queries:
    print(f'쿼리: {query}')
    doc, score = find_similar(query, embedded_documents)
    print(f'가장 유사한 문서: {doc}')
    print(f'유사도: {score:.2%}')
    print('-' * 100)

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 71.16%
----------------------------------------------------------------------------------------------------
쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 68.25%
----------------------------------------------------------------------------------------------------
쿼리: 컴퓨터가 이미지를 이해하는 방법은 무엇인가요?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 71.99%
----------------------------------------------------------------------------------------------------


# Hugging Face 임베딩 모델

Hugging Face의 임베딩 모델을 사용하려면 HuggingFaceEmbeddings 클래스를 활용하고 다양한 사전에 미리 훈련된 모델을 선택 가능하다.

장점  
&nbsp;&nbsp;&nbsp;▶ 다양한 사전 훈련 모델 중에서 선택 가능하다.  
&nbsp;&nbsp;&nbsp;▶ 오픈소스이므로 무료로 사용 가능하다.  
&nbsp;&nbsp;&nbsp;▶ 훈련된 모델을 다운받기 때문에 로컬에서 실행 가능하여 개인정보 보호에 유리하다. `C:\Users\사용자계정\.cache\huggingface\hub`에 다운된다.  
단점  
&nbsp;&nbsp;&nbsp;▶ 모델에 따라 성능 차이가 있다.  
&nbsp;&nbsp;&nbsp;▶ 로컬 실행 시 하드웨어 자원이 필요하다.  
&nbsp;&nbsp;&nbsp;▶ 일부 모델은 큰 용량을 차지할 수 있다(실행 속도가 느릴 수 있다).
<br><br>

<img src="huggingFaceEmbedding.png" width="1200" align="left" />

Hugging Face의 임베딩 모델을 사용하기 위해 HuggingFaceEmbeddings를 import 한다.

In [71]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

코드로 Hugging Face에 연결하기

`from huggingface_hub import login`  
`login(token='API key')`

.env 파일에서 Hugging Face 환경 변수 설정하기

`HF_TOKEN=API key`

위의 2가지 작업중 한 가지 이상 설정하지 않으면 아래와 같은 경고 메시지가 출력된다.  
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

In [72]:
# HuggingFaceEmbeddings 클래스의 생성자로 사용할 모델 이름을 넘겨셔 임베딩 객체를 만든다.
# 이 문장이 실행될 때 모델 파일이 내 컴퓨터로 자동 다운로드되며, 텍스트를 벡터로 변활할 준비를 마친 임베딩 객체가 생성된다.
embeddings_model = HuggingFaceEmbeddings(model_name='BAAI/bge-m3')
embeddings_model

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='BAAI/bge-m3', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

embed_query() 메소드는 단일 문장을 임베딩 모델을 통해 숫자로 변환한다.

In [73]:
embedded_query = embeddings_model.embed_query('인공지능이란 무엇인가요?')

print(len(embedded_query))
print(embedded_query)

1024
[-0.037039101123809814, -0.004837997257709503, 0.002937310840934515, -0.015514640137553215, -0.0009441987494938076, -0.04150169715285301, -0.0065745082683861256, 0.011289515532553196, 0.02161400392651558, 0.004928800743073225, -0.020340774208307266, 0.016905194148421288, -0.01287416834384203, 0.005518962629139423, 0.014988403767347336, 0.024228908121585846, 0.007369131315499544, -0.02804981917142868, -0.01493903435766697, -0.051851872354745865, -0.006705012172460556, -0.009251519106328487, -0.016980864107608795, 0.006491435691714287, 0.0529317706823349, 0.048137348145246506, -0.00806962139904499, -0.02317175082862377, 0.018142996355891228, -0.011328184977173805, -0.004240390844643116, -0.00635470449924469, -0.002271761419251561, 0.01432944554835558, -0.03563676029443741, -0.008155819028615952, -0.011798160150647163, -0.04542398452758789, -0.04073285311460495, 0.00221386575140059, -0.012132303789258003, 0.017896099016070366, -0.01914471760392189, -0.04192441701889038, 0.00091919541

embed_documents() 메소드는 여러 개의 문장을 임베딩 모델을 통해 숫자로 변환한다.

In [74]:
embedded_documents = embeddings_model.embed_documents(documents)

print(len(embedded_documents))
print(len(embedded_documents[0]))
print(embedded_documents)

5
1024
[[-0.039414435625076294, 0.008764945901930332, -0.012681664898991585, 0.0024531090166419744, -0.008944783359766006, -0.007383602671325207, -0.0053773014806210995, -0.009055851958692074, 0.032915208488702774, 0.006045480724424124, -0.02701302245259285, -0.027740929275751114, 0.0004440583288669586, 0.030136588960886, 0.017242908477783203, 0.017090419307351112, 0.025524815544486046, -0.021855991333723068, -0.01134131383150816, -0.05702253431081772, -0.00030160986352711916, 0.01354304514825344, -0.007450115401297808, 0.018574485555291176, 0.0028946709353476763, 0.008630674332380295, -0.0007445080555044115, -0.028904087841510773, 0.02072783373296261, -0.020500613376498222, 0.008069832809269428, -0.026754260063171387, 0.003963017370551825, -0.01630393974483013, -0.07406201213598251, -0.033650293946266174, -0.02387145347893238, -0.034550007432699203, -0.03478590399026871, 0.005482995882630348, -0.0500335656106472, -0.0028036341536790133, -0.02314690873026848, -0.07491140812635422, -0.0

## 유사도 검사

In [75]:
for query in queries:
    print(f'쿼리: {query}')
    doc, score = find_similar(query, embedded_documents)
    print(f'가장 유사한 문서: {doc}')
    print(f'유사도: {score:.2%}')
    print('-' * 100)

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 72.69%
----------------------------------------------------------------------------------------------------
쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 70.57%
----------------------------------------------------------------------------------------------------
쿼리: 컴퓨터가 이미지를 이해하는 방법은 무엇인가요?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 68.47%
----------------------------------------------------------------------------------------------------


# Ollama 임베딩 모델

Ollama의 임베딩 모델을 사용하려면 OllamaEmbeddings 클래스를 활용하고 로컬에서 실행되는 오픈소스 모델을 제공하며 다양한 모델을 선택 가능하다.

장점  
&nbsp;&nbsp;&nbsp;▶ 다양한 모델을 쉽게 설치하고 사용할 수 있다.  
&nbsp;&nbsp;&nbsp;▶ 오픈소스이므로 무료로 사용 가능하다.  
&nbsp;&nbsp;&nbsp;▶ 로컬에서 실행되어 개인정보 보호에 유리하다. `C:\Users\사용자계정\.ollama\models`에 다운된다.  
단점  
&nbsp;&nbsp;&nbsp;▶ 모델에 따라 성능 차이가 있다.  
&nbsp;&nbsp;&nbsp;▶ 로컬 실행 시 하드웨어 자원이 필요하다.  
&nbsp;&nbsp;&nbsp;▶ OpenAI나 일부 Hugging Face 모델에 비해 성능이 떨어질 수 있다.(특히, 한국어)

Ollama 임베딩 모델을 사용하려면 공식 웹 사이트에 접속하여 운영체제에 맞는 설치 파일을 다운로드해서 기본 설정으로 설치하면 된다.
<br><br>

<img src="ollama.png" width="1200" align="left" />

`ollama list` 명령을 실행하면 내 컴퓨터에 설치된 Ollama 모델 목록을 확인할 수 있다.
<img src="ollama_2.png" width="1200" align="left" />

`ollama pull 모델이름` 명령을 실행하면 Ollama 서버에서 지정한 모델을 내 컴퓨터로 다운받는다.
<img src="ollama_3.png" width="1200" align="left" />
<img src="ollama_4.png" width="1200" align="left" />

`ollama rm 모델이름` 명령을 실행하면 내 컴퓨터에 다운된 Ollama 모델을 제거한다.

Ollama 임베딩 모델을 사용하기 위해 OllamaEmbeddings 라이브러리를 import 한다.  
내 컴퓨터에서 실행중인 Ollama 서버(로컬 API)에 텍스트를 보내서 결과를 받아오는 방식으로 작동한다.

In [76]:
from langchain_ollama import OllamaEmbeddings

`nomic-embed-text`는 Nomic AI에서 개발한 오픈소스 고성능 문장/텍스트 임베딩 모델이다.  
`bge-m3`는 BAAI가 개발한 고성능 다국어 임베딩 모델을 Ollama 인프라에서 로컬 환경에 맞춰 GGUF 포맷으로 경량화·패키징한 모델이다.

In [77]:
# OllamaEmbeddings 클래스의 생성자로 사용할 모델 이름을 넘겨셔 임베딩 객체를 만든다.
# 모델 이름을 올바로 지정했는데 모델이 없다고 에러가 발생된다면 'ollama pull 모델이름' 명령을 실행해서 모델을 다운로드 받아야 한다.
# embeddings_model = OllamaEmbeddings(model='nomic-embed-text')
embeddings_model = OllamaEmbeddings(model='bge-m3')
embeddings_model

OllamaEmbeddings(model='bge-m3', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

embed_query() 메소드는 단일 문장을 임베딩 모델을 통해 숫자로 변환한다.

In [78]:
embedded_query = embeddings_model.embed_query('인공지능이란 무엇인가요?')

print(len(embedded_query))
print(embedded_query)

1024
[-0.036837306, -0.004901257, 0.0028993932, -0.015542105, -0.0009236714, -0.041678615, -0.0066977586, 0.011345492, 0.021567801, 0.0048781787, -0.020494865, 0.016804712, -0.012875808, 0.005440817, 0.014991537, 0.024282359, 0.007457474, -0.027947737, -0.015147981, -0.05180084, -0.006587092, -0.00919115, -0.017149175, 0.0065878658, 0.052873045, 0.04805971, -0.008183344, -0.023184633, 0.018107275, -0.0113778245, -0.0043240115, -0.0063718744, -0.0024370872, 0.014352622, -0.035450827, -0.008181477, -0.011819313, -0.045272134, -0.040645324, 0.0021609473, -0.012305348, 0.017717537, -0.019153874, -0.041932438, 0.0010296361, -0.039017648, -0.024438273, -0.024554454, -0.02218385, -0.0044951, 0.03152467, -0.048742488, 0.018029593, 0.025274457, 0.0023810337, 0.048528336, -0.008147021, 0.028082317, -0.07376654, -0.02071036, -0.0263737, -0.0075052455, -0.038743872, -0.017383734, 0.019272106, 0.063722305, 0.020412877, -0.011862565, -0.018534573, -0.040743165, 0.0024958854, 0.04152467, -0.057775315

embed_documents() 메소드는 여러 개의 문장을 임베딩 모델을 통해 숫자로 변환한다.

In [79]:
embedded_documents = embeddings_model.embed_documents(documents)

print(len(embedded_documents))
print(len(embedded_documents[0]))
print(embedded_documents)

5
1024
[[-0.039327543, 0.00869382, -0.012819257, 0.0023695487, -0.009030598, -0.007414694, -0.0055309543, -0.009002965, 0.03278059, 0.006018014, -0.027282547, -0.02791929, 0.0003824429, 0.030138217, 0.017149258, 0.017286131, 0.025451956, -0.021855853, -0.011497036, -0.05692895, -0.00026233925, 0.013527556, -0.007456929, 0.018568484, 0.0029138334, 0.008627365, -0.0008088525, -0.029066667, 0.020717213, -0.020586446, 0.008167912, -0.0268777, 0.0036514697, -0.01617703, -0.07386171, -0.033662938, -0.023805618, -0.03433129, -0.03461761, 0.0054978924, -0.05012636, -0.002977629, -0.023077039, -0.07500693, -0.011316185, -0.028952576, -0.034184914, -0.02544587, -0.061382174, 0.013581448, 0.024442885, -0.031426925, 0.0505767, 0.012991613, -0.06399184, 0.02731463, 0.0070647937, 0.010386289, -0.064060755, -0.0162003, -0.02071216, 0.030185914, -0.006335049, -0.013105099, -0.00068153784, 0.046958476, 0.0078419885, 0.029635437, -0.034817547, -0.03636436, 0.02149089, 0.017682996, -0.033513073, 0.013994

## 유사도 검사

In [80]:
for query in queries:
    print(f'쿼리: {query}')
    doc, score = find_similar(query, embedded_documents)
    print(f'가장 유사한 문서: {doc}')
    print(f'유사도: {score:.2%}')
    print('-' * 100)

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 72.71%
----------------------------------------------------------------------------------------------------
쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 70.52%
----------------------------------------------------------------------------------------------------
쿼리: 컴퓨터가 이미지를 이해하는 방법은 무엇인가요?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 68.39%
----------------------------------------------------------------------------------------------------


# 다국어 RAG 시스템 구축

하나의 다국어 지원 임베딩 모델로 하나의 벡터저장소에 다국어 문서를 한번에 저장한다. 임베딩 모델의 다국어 이해 능력이 중요하다.

# 교차 언어(cross-lingual)

## 다국어 문서 로드 및 전처리

특정 폴더에 저장된 파일들 중에서 규칙적인 이름의 패턴을 가진 파일들만 골라서 파일 목록을 만든다.

In [81]:
korean_txt_file = glob(os.path.join('./data', '*_KR.txt'))
english_txt_file = glob(os.path.join('./data', '*_EN.txt'))

print(korean_txt_file)
print(english_txt_file)

['./data\\리비안_KR.txt', './data\\테슬라_KR.txt']
['./data\\Rivian_EN.txt', './data\\Tesla_EN.txt']


텍스트 파일의 내용을 읽어들이기 위해서 TextLoader를 import 한다.

In [82]:
from langchain_community.document_loaders import TextLoader

읽어들일 텍스트 파일의 경로와 이름이 저장된 리스트 인수로 넘겨받아 텍스트 파일을 읽어오는 함수

In [83]:
def load_text_files(txt_files):
    data = []
    for txt_file in txt_files:
        loader = TextLoader(txt_file, encoding='utf-8')
        data += loader.load()
    return data

In [84]:
korean_data = load_text_files(korean_txt_file)
print(len(korean_data))
print(korean_data)

2
[Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다. 2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다. 주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다. 이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다. 리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다. 2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.\n\n리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.\n'), Document(metadata={'source': './data\\테슬라_KR.txt'}, page_content='테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.\n\n2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년

In [85]:
english_data = load_text_files(english_txt_file)
print(len(english_data))
print(english_data)

2
[Document(metadata={'source': './data\\Rivian_EN.txt'}, page_content="Founded in 2009 by MIT PhD graduate RJ Scaringe, Rivian is an innovative American electric vehicle manufacturer. Initially focused on autonomous electric cars from 2011, Rivian's significant growth phase began in 2015 with substantial investments, leading to the establishment of research facilities in Michigan and the Bay Area. To be closer to key suppliers, the company relocated its headquarters to Livonia, Michigan.\n\nRivian's early endeavor was the sports car R1 (originally named Avera), a mid-engine hybrid coupe with a 2+2 seating arrangement, designed by Peter Stevens. The car featured a modular capsule structure with easily replaceable body panels, and production was anticipated to start between late 2013 and early 2014. Rivian also considered various versions, including a diesel hybrid, a racing version named R1 GT for a Brazilian one-make series, a four-door sedan, and a crossover. Although a prototype hat

긴 텍스트 데이터(korean_data, english_data)를 구두점(.!?)을 경계로 AI 모델이 처리하기 좋은 크키(문서 조각, 청크)로 나눈다.

In [96]:
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    # 길이를 계산할 때 기준이 되는 모델을 지정한다. 모델마다 단어를 숫자로 치환하는 방식(토큰화)이 다르기 때문에 이를 명시해야 한다.
    model_name='text-embedding-3-small',
    # 나눠질 조각의 최대 길이를 100 토큰으로 제한한다.
    chunk_size=100,
    # 나눠질 조각들 사이에 겹치는 부분을 지정한다.
    chunk_overlap=0,
    # 텍스트를 자를 구분자를 지정한다.
    # separator=r'(?<=[.!?])\s+',
    separator=r'[.!?]\s+',
    # separator에 지정한 속성 값이 정규 표현식임을 의미한다.
    is_separator_regex=True,
    # 문장을 나눌 때 사용한 구분자를 결과물에 포함시킬지 여부를 지정한다.
    keep_separator=False
)

In [196]:
korean_docs = text_splitter.split_documents(korean_data)
print(len(korean_docs))
print([len(doc.page_content) for doc in korean_docs])
print(korean_docs[0])

13
[54, 73, 43, 76, 78, 80, 100, 54, 94, 93, 89, 56, 64]
page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다' metadata={'source': './data\\리비안_KR.txt'}


In [98]:
english_docs = text_splitter.split_documents(english_data)
print(len(english_docs))
print([len(doc.page_content) for doc in english_docs])
print(english_docs[0])

6
[432, 317, 372, 366, 324, 118]
page_content='Founded in 2009 by MIT PhD graduate RJ Scaringe, Rivian is an innovative American electric vehicle manufacturer[.!?]\s+Initially focused on autonomous electric cars from 2011, Rivian's significant growth phase began in 2015 with substantial investments, leading to the establishment of research facilities in Michigan and the Bay Area[.!?]\s+To be closer to key suppliers, the company relocated its headquarters to Livonia, Michigan' metadata={'source': './data\\Rivian_EN.txt'}


## 문서 임베딩 및 벡터저장소 저장

In [99]:
# 유료 클라우드 API 사용(OpenAI)
embeddings_openai = OpenAIEmbeddings(model='text-embedding-3-small')

# 오픈소스 모델을 내 컴퓨터 메모리에 직접 로드해서 사용
embeddings_huggingface = HuggingFaceEmbeddings(model_name='BAAI/bge-m3')

# 로컬 서버형 모델 사용
embeddings_ollama = OllamaEmbeddings(model='nomic-embed-text')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

조각낸 문서들을 Chroma라는 벡터저장소에 저장한다.  
단순히 텍스트를 저장하는 것이 아니라, 텍스트를 숫자로 저장하여 나중에 질문을 던지면 가장 관련있는 내용을 찾을 수 있다.

In [100]:
db_openai = Chroma.from_documents(
    # 저장할 문서 데이터를 지정한다.
    documents=korean_docs + english_docs,
    # 임베딩 모델을 지정한다.
    embedding=embeddings_openai,
    # 벡터저장소의 테이블 이름을 지정한다.
    collection_name='db_openai',
    # 벡터저장소가 저장될 실제 폴더를 지정한다.
    persist_directory='./chroma_db',
)

db_huggingface = Chroma.from_documents(
    documents=korean_docs + english_docs,
    embedding=embeddings_huggingface,
    collection_name='db_huggingface',
    persist_directory='./chroma_db',
)

db_ollama = Chroma.from_documents(
    documents=korean_docs + english_docs,
    embedding=embeddings_ollama,
    collection_name='db_ollama',
    persist_directory='./chroma_db',
)

In [101]:
print(db_openai._collection)
print(db_openai._collection.count())
print(db_huggingface._collection)
print(db_huggingface._collection.count())
print(db_ollama._collection)
print(db_ollama._collection.count())

Collection(name=db_openai)
19
Collection(name=db_huggingface)
19
Collection(name=db_ollama)
19


## OpenAI, HuggingFace, Ollama 각각의 임베딩 모델에 따른 RAG 성능 비교한다.

예전에는 ChatPromptTemplate이 `langchain.prompts` 모듈에 있었는데 버전이 올라가면서 `langchain_core.prompts` 모듈로 이동되었다.  
ChatPromptTemplate 사용자 입력을 AI가 이해하기 쉬운 대화형 프롬프트로 만들어주는 템플릿 도구로 SystemMessage(페르소나, 정체정, 역할)와 HumanMessage(질문)을 구조화해서 AI 공식 문서에서 권장하는 표준 방식으로 메시지를 구성한다.

In [103]:
# AI의 응답에서 메타데이터를 제외하고 순수 텍스트만 추출하기 위해 import 한다.
from langchain_core.output_parsers import StrOutputParser
# LCEL로 RAG 체인을 구성할 때 데이터 가공없이 사용자 입력을 프롬프트의 변수로 전달하기 위해 import 한다.
from langchain_core.runnables import RunnablePassthrough

프롬프트 템플릿 설정

AI가 답변할 때 지켜야 할 규칙(지침)을 정의한다.  
외부 지식을 차단하고 오직 제공된 {context} 변수에 입력된 내용 안에서만 답변하도록 강제하여 환각을 방지한다.

In [120]:
template = '''
다음 문맥(context)에만 근거하여 질문에 답변하세요.
외부 정보나 지식은 절대로 사용하지 마세요.
문맥(context)에 답이 포함되어 있지 않다면 '잘 모르겠습니다.'라고 답변하세요.

[문맥]
{context}

[질문]
{question}

[답변]

'''

template2 = '''
Answer the question based ONLY on the following context.
Do NOT use any outside information or knowledge.
If the context does not contain the answer, respond with 'I don't know.'

[Context]
{context}

[Question]
{question}

[Answer]
'''

prompt = ChatPromptTemplate.from_template(template)

벡터저장소에서 검색된 문서 조각들을 프롬프트의 {context} 변수에 넣어주기 위해서 하나의 문자열로 각 문서 조각 사이에 '\n\n'를 넣어서 연결하는 함수

In [121]:
def format_docs(docs):
    return '\n\n'.join([doc.page_content for doc in docs])

답변의 일관성을 높이고 창의성을 억제해서 사실에 기반한 일관된 답변을 하도록 RAG에서 사용할 AI 모델을 설정한다.

In [122]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

벡터검색기(retriever)에서 프롬프트의 {context} 변수에 넣어줄 문서 조각을 검색하고 RAG 체인을 생성하는 함수

In [123]:
def create_rag_chain(vectorstore):
    # 벡터저장소를 벡터검색기로 변환하고 유사도가 높은 순서로 2개의 문서 조각을 가져온다.
    retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
    # LCEL로 RAG 체인을 생성해서 리턴한다.
    return (
        # 프롬프트의 변수({context}, {question})로 전달할 데이터
        # {context}로는 벡터검색기(retriever)가 검색한 결과를 format_docs 함수에 전달해서 한 개로 연결한 문자열을 전달한다.
        # {question}로는 RunnablePassthrough() 객체를 이용해서 사용자가 입력한 질문을 그대로 전달한다.
        {'context': retriever | format_docs, 'question': RunnablePassthrough()}
        # 전단계의 'context'와 'question'을 프롬프트의 변수({context}, {question})에 넣어서 프롬프트를 완성한다.
        | prompt
        # 프롬프트 AI에게 던져서 응답을 받는다.
        | llm
        # AI에게 응답받은 결과에서 메타데이터를 제외하고 문자열만 추출한다.
        | StrOutputParser()
    )

OpenAI, HuggingFace, Ollama 각각의 임베딩 모델을 사용한 RAG 체인을 생성한다.

In [124]:
# OpenAI 임베딩 모델로 만든 벡터저장소로 RAG 체인을 생성한다.
rag_chain_openai = create_rag_chain(db_openai)

# HuggingFace 임베딩 모델로 만든 벡터저장소로 RAG 체인을 생성한다.
rag_chain_huggingface = create_rag_chain(db_huggingface)

# Ollama 임베딩 모델로 만든 벡터저장소로 RAG 체인을 생성한다.
rag_chain_ollama = create_rag_chain(db_ollama)

OpenAI, HuggingFace, Ollama 각각의 임베딩 모델을 사용한 RAG 체인에 똑같은 질문을 던져서 성능을 테스트 한다.

한글 쿼리에 대한 성능 테스트

In [125]:
query_ko = '테슬라 창업자는 누구인가요?'

# OpenAI 임베딩 모델로 만든 벡터저장소로 생성 RAG 체인을 실행한다.
output_openai = rag_chain_openai.invoke(query_ko)
print(f'OpenAI: {output_openai}')

# HuggingFace 임베딩 모델로 만든 벡터저장소로 생성 RAG 체인을 실행한다.
output_huggingface = rag_chain_huggingface.invoke(query_ko)
print(f'HuggingFace: {output_huggingface}')

# Ollama 임베딩 모델로 만든 벡터저장소로 생성 RAG 체인을 실행한다.
output_ollama = rag_chain_ollama.invoke(query_ko)
print(f'Ollama: {output_ollama}')

OpenAI: 마틴 에버하드와 마크 타페닝이 테슬라의 창립자입니다.
HuggingFace: 마틴 에버하드와 마크 타페닝입니다.
Ollama: 잘 모르겠습니다.


영어 쿼리에 대한 성능 테스트

In [126]:
query_en = 'Who is the founder of Tesla?'

# OpenAI
output_openai = rag_chain_openai.invoke(query_en)
print(f'OpenAI: {output_openai}')

# HuggingFace
output_huggingface = rag_chain_huggingface.invoke(query_en)
print(f'HuggingFace: {output_huggingface}')

# Ollama
output_ollama = rag_chain_ollama.invoke(query_en)
print(f'Ollama: {output_ollama}')

OpenAI: Tesla는 Martin Eberhard와 Marc Tarpenning에 의해 2003년에 설립되었습니다.
HuggingFace: Tesla는 Martin Eberhard와 Marc Tarpenning에 의해 2003년에 설립되었습니다.
Ollama: Tesla는 Martin Eberhard와 Marc Tarpenning에 의해 2003년에 설립되었습니다.


# 언어 감지 및 검색 라우팅

사용자의 쿼리에 사용된 언어를 감지하여 해당 언어 문서가 저장되어 있는 벡터저장소에 검색해서 쿼리와 문서 간에 동일한 언어를 기반으로 처리한다.

## 언어별 벡터저장소 생성

서로 다른 언어(한국어, 영어)로 된 문서들을 각각 최적화된 임베딩 모델을 사용해서 별도의 벡터저장소를 생성한다.

In [129]:
# HuggingFace 임베딩 모델을 사용해서 한국어 문서 벡터저장소 생성한다.
db_korean = Chroma.from_documents(
    documents=korean_docs,
    embedding=embeddings_huggingface,
    collection_name='db_korean',
    persist_directory='./chroma_db',
)

# Ollama 임베딩 모델을 사용해서 영어 문서 벡터저장소 생성한다.
db_english = Chroma.from_documents(
    documents=english_docs,
    embedding=embeddings_ollama,
    collection_name='db_english',
    persist_directory='./chroma_db',
)

In [130]:
print(db_korean._collection)
print(db_korean._collection.count())
print(db_english._collection)
print(db_english._collection.count())

Collection(name=db_korean)
13
Collection(name=db_english)
6


## 언어에 따른 RAG 성능 비교한다.

질문의 언어를 감지한 뒤, 해당 언어에 맞춰 미리 준비된 벡터저장소와 체인으로 연결하는 라우팅 시스템이다.

문장의 언어를 판별하기 위해 detect를 import 한다.

In [149]:
from langdetect import detect

한국어를 사용하는 RAG 체인과 영어를 사용하는 RAG 체인을 만든다.

In [150]:
rag_chain_korean = create_rag_chain(db_korean)
rag_chain_english = create_rag_chain(db_english)

질문에 사용된 언어가 무엇인지 판단해서 판단된 언어를 사용하는 RAG 체인을 실행하는 함수

In [184]:
def route_rag_chain(query):
    # detect() 함수의 인수로 질문(query)을 넘겨서 입력된 질문이 어떤 언어인지 판단한다.
    language = detect(query)
    # print(language)
    if language.upper() == 'KO':
        # 쿼리가 한글이면 한국어를 사용하는 RAG 체인을 리턴한다.
        return rag_chain_korean.invoke(query)
    elif language.upper() == 'EN':
        # 쿼리가 영어면 영를 사용하는 RAG 체인을 리턴한다.
        return rag_chain_english.invoke(query)
    else:
        return '한글과 영어 이외의 언어는 처리할 수 없습니다.'
    return language

In [185]:
output_korean = route_rag_chain(query_ko)
print(output_korean)
output_english = route_rag_chain(query_en)
print(output_english)
output_japan = route_rag_chain('テスラの創業者は誰ですか？')
print(output_japan)
output_china = route_rag_chain('特斯拉的创始人是谁？')
print(output_china)

마틴 에버하드와 마크 타페닝입니다.
Tesla는 Martin Eberhard와 Marc Tarpenning에 의해 2003년에 설립되었습니다.
한글과 영어 이외의 언어는 처리할 수 없습니다.
한글과 영어 이외의 언어는 처리할 수 없습니다.


# Gradio 챗봇 - 히스토리 추가하기

과거 대화 기록을 LangChain 객체로 변환하고 RAG 시스템과 연동한다.

In [161]:
import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

In [186]:
def answer_history(message, history):
    # LangChain 모델이 이해할 수 있는 대화내용 메시지 객체들을 저장할 빈 리스트를 선언한다.
    history_message = []
    
    # 이전 대화 메시지를 처리한다.
    '''
    과거의 gradio(3.X 버전 까지)
    이전 대화 내용(history)이 [[유저질문, AI답변], [유저질문, AI답변], ...]와 같은 구조의 리스트 형태여서 아래와 같은 반복문을 사용했다.
    for human, ai in history: 형태의 반복문을 사용했다.
        history_message.append(HumanMessage(content=human))
        history_message.append(AIMessage(content=ai))
        
    현재의 gradio(4.X 버전 부터)
    이전 대화 내용이 [{'role': 'user', 'content': '...'}, {'role': 'assistant', 'content': '...'}] 구조의 리스트 형태이다.
    for human, ai in history: 형태의 반복문을 사용하면 에러가 발생되므로 아래와 같은 반복문을 사용한다.
    '''
    for msg in history:
        # print(msg)
        if msg['role'] == 'user':
            # 사용자 질문은 HumanMessage 객체로 만들어서 대화 내용을 기억하는 리스트에 추가한다.
            history_message.append(HumanMessage(content=msg['content']))
        elif msg['role'] == 'assistant':
            # AI의 응답은 AIMessage 객체로 만들어서 대화 내용을 기억하는 리스트에 추가한다.
            history_message.append(AIMessage(content=msg['content']))
    
    # 현재 질문으로 RAG 실행
    # 현재 질문은 HumanMessage 객체로 만들어서 대화 내용을 기억하는 리스트에 추가한다.
    history_message.append(HumanMessage(content=message))
    # 현재 질문으로 질문에 사용된 언어가 무엇인지 판단해서 판단된 언어를 사용하는 RAG 체인을 실행하는 함수를 실행한다. 현재 질문에 대한 답변
    response = route_rag_chain(message)
    
    # 이전 대화 내용을 context로 넘겨 최종 답변을 생성한다.
    final_answer = llm.invoke(
        # history_message[:-1]는 현재 질문을 제외한 이전 대화들을 의미한다.
        # [AIMessage(content=response)]는 현재 질문에 대해서 벡터검색기에서 찾아낸 정보를 의미한다.
        # [HumanMessage(content=message)]는 현재 질문을 의미한다.
        # 이렇게 구성하면 과거 대화(history_message[:-1])에 RAG로 찾은 정보([AIMessage(content=response)])를 모두 고려해서 사용자 질문에 대한 가장 완벽한
        # 답변을 얻어올 수 있다.
        history_message[:-1] + [AIMessage(content=response)] + [HumanMessage(content=message)]
    )
    return final_answer.content

In [187]:
chatbot = gr.ChatInterface(fn=answer_history, title='QA Bot')
chatbot.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


테슬라는 2020년에 설립되었습니다. 이때 회장은 누구인가요?  
내가 테슬라가 언제 설립되었다고 이야기했는지 기억해?

In [188]:
chatbot.close()

Closing server running on port: 7871
